In [1]:
import pandas as pd

In [2]:
goals_df = pd.read_csv("goals.csv")

goals_df.head(10)

,Team,Player,GameDay1,GameDay2,GameDay3,GameDay4,GameDay5,GameDay6,GameDay7,GameDay8,GameDay9,GameDay10,GameDay11,GameDay12
0,North 60,Leo,1.0,NaN,3.0,1.0,1.0,NaN,2.0,1.0,NaN,1.0,,2.0
1,Timbits,Kade,NaN,1.0,1.0,NaN,NaN,NaN,1.0,1.0,NaN,NaN,4,3.0
2,North 60,Ben Warburton,NaN,1.0,NaN,1.0,NaN,2.0,NaN,1.0,NaN,2.0,1,1.0
3,Idolos,Gabe,NaN,2.0,2.0,NaN,NaN,NaN,1.0,NaN,NaN,2.0,NaN,2.0
4,Riverdale FC,Harry,NaN,NaN,NaN,NaN,2.0,NaN,NaN,4.0,1.0,1.0,1,NaN
5,North 60,Ben Kishchuk,NaN,1.0,1.0,NaN,1.0,2.0,NaN,1.0,NaN,NaN,NaN,NaN
6,North 60,Cody Reaume CJ,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,2.0
7,Timbits,Daniel Philips Freedman,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN
8,Riverdale FC,Jesse,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,1,NaN
9,Idolos,Mark Chandler,1.0,1.0,1.0,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN


In [3]:
goal_format_df = pd.read_csv("goal_format.csv")
goal_format_df.head(10)

,Team1,Team2,gameday,game_time
0,Idolos,Riverdale FC,GameDay1,early
1,North 60,Timbits,GameDay1,late
2,Idolos,North 60,GameDay2,early
3,Riverdale FC,Timbits,GameDay2,late
4,Idolos,Timbits,GameDay3,early
5,Riverdale FC,North 60,GameDay3,late
6,Riverdale FC,Idolos,GameDay4,early
7,North 60,Timbits,GameDay4,late
8,Riverdale FC,Timbits,GameDay5,early
9,North 60,Idolos,GameDay5,late


In [5]:
long_df = goals_df.melt(
    id_vars=["Team", "Player"],   # keep these columns as they are
    var_name="game_day",          # new column name for the old headers
    value_name="goal_num"         # new column name for the values
)
long_df["goal_num"] = pd.to_numeric(long_df["goal_num"], errors="coerce")

long_df = long_df.dropna(subset=["goal_num"]) # drop rows where player had no goals

long_df["goal_num"] = long_df["goal_num"].astype(int) # convert goal_num to int (so it works for next function)

expanded_df = long_df.loc[long_df.index.repeat(long_df["goal_num"])].copy() # code to create new rows


expanded_df.reset_index(drop=True, inplace=True) # reset index (because we dropped rows and added new ones)

expanded_df["goal_num"] = expanded_df.groupby(["game_day", "Player"]).cumcount() + 1 # adding goal counter for multiple goals by player in same game

# print(expanded_df)

# Merge where the player's team is Team1
merge1 = expanded_df.merge(
    goal_format_df,
    left_on=["Team", "game_day"], # if they have the same team and game_day then add goal_format row to expanded_df row
    right_on=["Team1", "gameday"], # otherwise just add goal_format as NaN
    how="left"
)

# Merge where the player's team is Team2
merge2 = expanded_df.merge(
    goal_format_df,
    left_on=["Team", "game_day"],
    right_on=["Team2", "gameday"],
    how="left"
)

# Combine results (stack them together)
merged = pd.concat([merge1, merge2], ignore_index=True) # note that own goals did not make it, thats okay I guess

# Drop the rows where it didn’t match (NaN game_time)
merged = merged.dropna(subset=["game_time"])

final_df = merged[["game_day", "Player", "Team", "goal_num", "game_time"]] # pick only columns I want

#sorting
order = [f"GameDay{i}" for i in range(1, 18)]

# Convert GameDay to a categorical type with this order
final_df["game_day"] = pd.Categorical(final_df["game_day"], categories=order, ordered=True)

# Sort by GameDay, then maybe by Player or Goal number if you want
final_df = final_df.sort_values(by=["game_day", "game_time", "Team", "Player", "goal_num"])

print(final_df)

# sort by game day, then Semis, then Finals

final_df.to_csv('WFL2223_goals.csv', index=False)





      game_day          Player          Team  goal_num game_time
3     GameDay1             Dom        Idolos         1     early
2     GameDay1   Mark Chandler        Idolos         1     early
5     GameDay1           Tyler        Idolos         1     early
151   GameDay1            Huey  Riverdale FC         1     early
152   GameDay1            Huey  Riverdale FC         2     early
..         ...             ...           ...       ...       ...
282  GameDay12  Cody Reaume CJ      North 60         1      late
283  GameDay12  Cody Reaume CJ      North 60         2      late
285  GameDay12            Lars      North 60         1      late
274  GameDay12             Leo      North 60         1      late
275  GameDay12             Leo      North 60         2      late

[139 rows x 5 columns]


/var/folders/_m/j0k7jtkx50j8m1p216jbjf080000gn/T/ipykernel_15216/2199219487.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df["game_day"] = pd.Categorical(final_df["game_day"], categories=order, ordered=True)
